In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — no GPU detected')

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'efficientnet_pytorch', 'retinaface_pytorch', 'opencv-python',
    'tqdm', 'scikit-learn', 'pandas', 'gdown', 'pretrainedmodels', 'albumentations',
])

In [ ]:
import sys, subprocess
from pathlib import Path

SBI_DIR = Path('/content/SelfBlendedImages')
if not SBI_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1',
        'https://github.com/mapooon/SelfBlendedImages.git', str(SBI_DIR)])
sys.path.insert(0, str(SBI_DIR / 'src'))
print('SBI src ready')

In [ ]:
import gdown
from pathlib import Path

WEIGHTS_DIR = Path('/content/weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

weights_files = {
    'FFraw': ('12sLyqBp0VFwdpA-oZLdIOkOTkz_ZnIhV', WEIGHTS_DIR / 'FFraw.tar'),
    'FFc23': ('1X0-NYT8KPursLZZdxduRQju6E52hauV0',  WEIGHTS_DIR / 'FFc23.tar'),
}
for name, (fid, dest) in weights_files.items():
    if not dest.exists():
        print(f'Downloading {name}...')
        gdown.download(id=fid, output=str(dest), quiet=False)
    else:
        print(f'{name} already cached.')

weights_paths = [str(v[1]) for v in weights_files.values()]
print('Weights ready:', weights_paths)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_DIR = Path('/content/drive/MyDrive/deepfake videos/archive/DFD_manipulated_sequences/DFD_manipulated_sequences')

video_list = sorted(INPUT_DIR.glob('*.mp4'))
assert video_list, f'No .mp4 files found in {INPUT_DIR}'
print(f'{len(video_list)} video(s) found')

In [ ]:
import numpy as np, torch, csv
from datetime import datetime
from tqdm.notebook import tqdm
from retinaface.pre_trained_models import get_model
from model import Detector
from inference.preprocess import extract_frames

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FRAMES = 32

def load_model(path):
    m = Detector().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE)['model'])
    m.eval()
    return m

def score_faces(faces, idx_list, model):
    if not faces:
        return 0.5
    with torch.no_grad():
        preds = model(torch.from_numpy(np.stack(faces)).to(DEVICE).float() / 255).softmax(1)[:, 1]
    buckets = {}
    for idx, p in zip(idx_list, preds.tolist()):
        buckets.setdefault(idx, []).append(p)
    return float(np.mean([max(v) for v in buckets.values()]))

print('Loading models...')
models = [load_model(p) for p in weights_paths]
face_detector = get_model('resnet50_2020-07-20', max_size=2048, device=DEVICE)
face_detector.eval()
print(f'Ready on {DEVICE}')

In [ ]:
OUTPUT_DIR = Path('/content/results')
OUTPUT_DIR.mkdir(exist_ok=True)
out_file = OUTPUT_DIR / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with open(out_file, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['video', 'score', 'prediction'])
    for video in tqdm(video_list, desc='Inferring'):
        faces, idx_list = extract_frames(str(video), N_FRAMES, face_detector)
        score = max(score_faces(faces, idx_list, m) for m in models)
        prediction = 'FAKE' if score >= 0.5 else 'real'
        w.writerow([video.name, f'{score:.4f}', prediction])
        f.flush()
        tqdm.write(f'  {video.name:<55} {prediction:<6}  ({score:.4f})')

print(f'Done. Results -> {out_file}')

In [ ]:
from google.colab import files
from pathlib import Path

# pick up out_file from the inference cell, or fall back to the latest CSV in /content/results/
if 'out_file' not in dir():
    results = sorted(Path('/content/results').glob('results_*.csv'))
    assert results, "No results CSV found — run the inference cell first"
    out_file = results[-1]

files.download(str(out_file))
print('Downloaded:', out_file)
